In [1]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("../data/raw/PFE/XNAS-20260425-UEJBDCM7RR/xnas-itch-20250401-20250430.mbp-10.PFE.csv")
df.head()

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol
0,1743494400011096690,1743494400010927943,10,2,12497,A,N,0,25180000000,500,...,0,0,0,9223372036854775807,9223372036854775807,0,0,0,0,PFE
1,1743494400011133612,1743494400010965227,10,2,12497,A,A,0,25420000000,500,...,0,0,0,9223372036854775807,9223372036854775807,0,0,0,0,PFE
2,1743494400012006920,1743494400011840629,10,2,12497,A,B,0,25250000000,100,...,0,0,0,9223372036854775807,9223372036854775807,0,0,0,0,PFE
3,1743494400064927961,1743494400064759616,10,2,12497,A,B,1,25200000000,500,...,0,0,0,9223372036854775807,9223372036854775807,0,0,0,0,PFE
4,1743494400064955767,1743494400064789669,10,2,12497,C,A,0,25420000000,500,...,0,0,0,9223372036854775807,9223372036854775807,0,0,0,0,PFE


## Data pre-process
The data is very messy here. So we only trust the snapshot change in time:
* for each `ts_recv` (each snapshot), it should have one ended line with `flags` = 128 or 130 


What my preprocess do is to check the change of snapshot by `book_diff` \
and reconstruct the events and book state I want in the late for loop


In [6]:
from dataclasses import dataclass
from typing import Literal

N_LEVELS = 4

@dataclass
class BookChange:
    side:        Literal['bid', 'ask']
    level:       int   # 0-indexed position in the book
    price:       int
    size_delta:  int   # positive = added, negative = removed

def parse_side(row, side: str, n_levels: int) -> dict[int, tuple[int, int]]:
    """Extract {price: (level, size)} for one side, skipping empty levels."""
    best = row[f'{side}_px_00']
            
    return {
        int(row[f'{side}_px_{i:02d}']): (int(abs(row[f'{side}_px_{i:02d}']-best)*1e-7)+1, int(row[f'{side}_sz_{i:02d}']))
        for i in range(n_levels)
        if abs(row[f'{side}_px_{i:02d}']-best) < (n_levels-0.5)*1e7 
        # only consider the change in first four queue level (each level is a tick away from the previous one)
        # minus 0.5 to prevent python overflow
    }

def book_diff(old_row, new_row, n_levels: int = N_LEVELS) -> list[BookChange]:
    changes = []

    for side in ('bid', 'ask'):
        old_book = parse_side(old_row, side, n_levels)
        new_book = parse_side(new_row, side, n_levels)

        for price in old_book.keys() | new_book.keys():
            old_level, old_size = old_book.get(price, (None, 0))
            new_level, new_size = new_book.get(price, (None, 0))

            if old_size != new_size:
                # Prefer the new level; fall back to old if price was removed
                level = new_level if new_level is not None else old_level
                changes.append(BookChange(side, level, price, new_size - old_size))

    return changes

The reason why I directly detect the snapshot change is becasue there might be **multiple** change between snapshots\
But the dataset from databento are not availible to detect it

In [12]:
from dataclasses import asdict
state_a = df.iloc[359].to_dict()
state_b = df.iloc[360].to_dict()

changes = book_diff(state_a, state_b)  # → [BookChange('ask', 13, -10)]
event_df = pd.DataFrame([asdict(c) for c in changes])
event_df.head()

,side,level,price,size_delta
0,bid,1,25290000000,100
1,ask,1,25310000000,200


In [13]:
df.iloc[100:110]

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol
100,1743494450723703389,1743494450723537273,10,2,12497,C,B,0,25280000000,500,...,13,3,2,24390000000,27390000000,40,3,1,1,PFE
101,1743494450723718268,1743494450723551607,10,2,12497,A,B,1,25260000000,500,...,13,1,2,24500000000,27390000000,22,3,3,1,PFE
102,1743494450724243336,1743494450724077203,10,2,12497,C,B,0,25280000000,100,...,13,3,2,24390000000,27390000000,40,3,1,1,PFE
103,1743494450724349412,1743494450724183201,10,2,12497,A,B,0,25270000000,92,...,13,1,2,24500000000,27390000000,22,3,3,1,PFE
104,1743494450724886535,1743494450724720639,10,2,12497,C,A,0,25340000000,500,...,13,1,2,24500000000,27390000000,22,3,3,1,PFE
105,1743494450724900721,1743494450724733960,10,2,12497,A,A,1,25360000000,500,...,13,1,2,24500000000,27390000000,22,3,3,1,PFE
106,1743494450725383770,1743494450725217713,10,2,12497,C,A,0,25340000000,92,...,3,1,1,24500000000,27830000000,22,1,3,1,PFE
107,1743494450725443417,1743494450725277394,10,2,12497,A,A,0,25350000000,109,...,13,1,2,24500000000,27390000000,22,3,3,1,PFE
108,1743494451740646673,1743494451740480257,10,2,12497,A,B,0,25280000000,500,...,13,1,2,24500000000,27390000000,22,3,3,1,PFE
109,1743494451803398806,1743494451803232551,10,2,12497,C,B,1,25270000000,92,...,13,3,2,24390000000,27390000000,40,3,1,1,PFE


In [29]:
from dataclasses import asdict

previous = None
events = []  # collect dicts, concat once at the end
states = []

for ts, temp_df in df.iloc[100:110].groupby('ts_recv'):
    now = temp_df.iloc[-1].to_dict()
    
    ask = parse_side(now, 'ask', n_levels=N_LEVELS)
    bid = parse_side(now, 'bid', n_levels=N_LEVELS)
    
    best_ask = min(ask.keys())
    best_bid = max(bid.keys())
    
    if previous is not None:
        changes = book_diff(previous, now)        
        if changes:  # skip empty diffs
            reduce_reason = 'T' if 'T' in temp_df['action'].values else 'C'
            
            is_create = (best_ask < best_ask_p) or (best_bid > best_bid_p)
            
            for c in changes:
                d = asdict(c)
                if d['size_delta'] > 0:
                    increase_reason = 'E' if (is_create and (d["level"]==1)) else 'A' #Detection for create event
                    d['action'] = increase_reason
                else:
                     d['action'] = reduce_reason
                d['ts'] = ts
                d['size_delta'] = abs(d['size_delta'])
                events.append(d)

            level_to_size_ask = {level: size for level, size in ask.values()}
            level_to_size_bid = {-level: size for level, size in bid.values()}

            state = pd.Series(level_to_size_bid | level_to_size_ask).reindex(range(-4, 4+1), fill_value=0)

            state['spread'] = int((best_ask-best_bid)*1e-7)
            state['imb'] = (state[1]-state[-1])/(state[1]+state[-1])
            state['best_px'] = (best_ask+best_bid)/2*1e-9
            
            states.append(state)

        
    previous = temp_df.iloc[-1].to_dict()  # always update, even on first iter
    best_ask_p = best_ask 
    best_bid_p = best_bid
    
event_df = pd.DataFrame(events, columns=['ts', 'side', 'level', 'price', 'size_delta', 'action'])
states_df = pd.DataFrame(states)
states_df['ts'] = df.iloc[100:110]['ts_recv'].drop_duplicates(ignore_index=True)
states_df.drop(columns=[0], inplace=True)

In [30]:
event_df

,ts,side,level,price,size_delta,action
0,1743494450723718268,bid,3,25260000000,500,A
1,1743494450724243336,bid,1,25280000000,100,C
2,1743494450724349412,bid,1,25270000000,92,E
3,1743494450724886535,ask,1,25340000000,500,C
4,1743494450724900721,ask,3,25360000000,500,A
5,1743494450725383770,ask,1,25340000000,92,C
6,1743494450725443417,ask,1,25350000000,109,E
7,1743494451740646673,bid,2,25260000000,500,C
8,1743494451740646673,bid,1,25280000000,500,E
9,1743494451803398806,bid,2,25270000000,92,C


In [31]:
states_df

,-4,-3,-2,-1,1,2,3,4,spread,imb,best_px,ts
0,100.0,500.0,0.0,100.0,592.0,0.0,30.0,0.0,6.0,0.710983,25.310,1743494450723703389
1,0.0,0.0,100.0,500.0,592.0,0.0,30.0,0.0,8.0,0.084249,25.300,1743494450723718268
2,0.0,100.0,500.0,92.0,592.0,0.0,30.0,0.0,7.0,0.730994,25.305,1743494450724243336
3,0.0,100.0,500.0,92.0,92.0,0.0,30.0,0.0,7.0,0.000000,25.305,1743494450724349412
4,0.0,100.0,500.0,92.0,92.0,0.0,530.0,0.0,7.0,0.000000,25.305,1743494450724886535
5,0.0,100.0,500.0,92.0,530.0,0.0,0.0,0.0,9.0,0.704180,25.315,1743494450724900721
6,0.0,100.0,500.0,92.0,109.0,530.0,0.0,0.0,8.0,0.084577,25.310,1743494450725383770
7,100.0,0.0,92.0,500.0,109.0,530.0,0.0,0.0,7.0,-0.642036,25.315,1743494450725443417
8,100.0,0.0,0.0,500.0,109.0,530.0,0.0,0.0,7.0,-0.642036,25.315,1743494451740646673


### Discretelize imbalance

In [27]:
def get_imbalance_bin(series):
    v = series.to_numpy()
    
    # Divide by 0.1 and round to fix floating-point precision issues
    # e.g., -0.1 / 0.1 could be -0.9999... instead of -1.0
    scaled = np.round(v / 0.1, 8)

    result = np.where(
        v == 0,                          # bin 0: exactly zero
        0,
        np.where(
            v < 0,
            np.floor(scaled).astype(int),  # negative: [0.1*i, 0.1*(i+1))
            np.ceil(scaled).astype(int)    # positive: (0.1*(i-1), 0.1*i]
        )
    )
    return result

In [28]:
states_df['imb'] = get_imbalance_bin(states_df['imb'])
states_df

,-4,-3,-2,-1,0,1,2,3,4,spread,imb,best_px,ts
0,100.0,500.0,0.0,100.0,0.0,592.0,0.0,30.0,0.0,6.0,8,25.310,1743494450723703389
1,0.0,0.0,100.0,500.0,0.0,592.0,0.0,30.0,0.0,8.0,1,25.300,1743494450723718268
2,0.0,100.0,500.0,92.0,0.0,592.0,0.0,30.0,0.0,7.0,8,25.305,1743494450724243336
3,0.0,100.0,500.0,92.0,0.0,92.0,0.0,30.0,0.0,7.0,0,25.305,1743494450724349412
4,0.0,100.0,500.0,92.0,0.0,92.0,0.0,530.0,0.0,7.0,0,25.305,1743494450724886535
5,0.0,100.0,500.0,92.0,0.0,530.0,0.0,0.0,0.0,9.0,8,25.315,1743494450724900721
6,0.0,100.0,500.0,92.0,0.0,109.0,530.0,0.0,0.0,8.0,1,25.310,1743494450725383770
7,100.0,0.0,92.0,500.0,0.0,109.0,530.0,0.0,0.0,7.0,-7,25.315,1743494450725443417
8,100.0,0.0,0.0,500.0,0.0,109.0,530.0,0.0,0.0,7.0,-7,25.315,1743494451740646673


### Filter trading hours

In [32]:
import pandas_market_calendars as mcal

def filter_trading_hours(df: pd.DataFrame, ts_col: str = "ts_recv") -> pd.DataFrame:
    """
    filter the first and last 30 min of each trading day
    remaining: 10:00~15:00 each trading day
    """
    # Convert nanosecond UTC to ET
    ts_et = pd.to_datetime(df[ts_col], unit="ns", utc=True).dt.tz_convert("America/New_York")
    
    # Get the date range covered by the data
    start_date = ts_et.dt.date.min()
    end_date   = ts_et.dt.date.max()
    
    # Get the NYSE trading calendar for that range
    nyse = mcal.get_calendar("NYSE")
    schedule = nyse.schedule(
        start_date=start_date.strftime("%Y-%m-%d"),
        end_date=end_date.strftime("%Y-%m-%d")
    )
    
    # schedule gives you market_open and market_close in UTC for each trading day
    # but we will use fixed 9:30-15:30 ET since regular session times don't change
    trading_dates = set(schedule.index.date)
    
    # Build mask: must be a trading day AND within session hours
    date_only = ts_et.dt.date
    time_only = ts_et.dt.time
    
    market_open  = pd.Timestamp("10:00:00").time()
    market_close = pd.Timestamp("15:30:00").time()
    
    is_trading_day   = date_only.apply(lambda d: d in trading_dates)
    is_trading_hours = (time_only >= market_open) & (time_only <= market_close)
    
    mask = is_trading_day & is_trading_hours
    
    return df[mask].reset_index(drop=True)

In [33]:
filter_trading_hours(states_df, 'ts')

,-4,-3,-2,-1,1,2,3,4,spread,imb,best_px,ts
